# 02a · Slice-level curation (reproducible, training-only)
Rebuilds the manual 'remove infested slices without visible signs' step as an **objective, logged** procedure. A per-slice *damage score* (fraction of interior fruit pixels that are cavity-dark) is thresholded at a percentile of **training control** slices; among **training infested** fruit only, slices below threshold are dropped from training. Validation/test are never modified.

Run AFTER `01_partitioning`. Produces a curated slice tree + an editable manifest for human review (human-in-the-loop). Set `CFG.curated_root` to the output path before running `02_train`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()/'nbpkg'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from config import CFG
import dataset as ds, slice_curation as sc
CFG.out_dir.mkdir(parents=True, exist_ok=True)

## Step 1 — load fruits and the train/val/test split

In [ ]:
fruits = ds.build_fruit_index(CFG.data_root, fruit_key=CFG.fruit_key)[0]
by={f.fruit_id:f for f in fruits}
split=pd.read_csv(CFG.out_dir/'split_single.csv').set_index('fruit_id')['split']
train=[by[i] for i in split[split=='train'].index]
tr_ctrl=[f for f in train if f.label==0]; tr_inf=[f for f in train if f.label==1]
print(f'train fruits: {len(train)}  (control={len(tr_ctrl)}, infested={len(tr_inf)})')

## Step 2 — calibrate the damage threshold on TRAIN CONTROL slices only

In [ ]:
thr, ctrl_scores = sc.calibrate_threshold(tr_ctrl, percentile=CFG.curation_percentile)
print(f'threshold = {thr:.4f}  ({CFG.curation_percentile}th pct of {len(ctrl_scores)} control slices)')

## Step 3 — score distributions (control vs infested training slices)
You want the infested distribution to have a tail above the threshold. If the two overlap almost entirely, raise/lower `curation_percentile` or tune the score parameters (`dark_frac_of_pulp`, `central_band_frac`) in slice_curation.py.

In [ ]:
inf_scores=np.array([s for f in tr_inf for _,s in sc.score_fruit(f)])
plt.figure(figsize=(7,3))
plt.hist(ctrl_scores,bins=40,alpha=.6,label='control',density=True)
plt.hist(inf_scores,bins=40,alpha=.6,label='infested',density=True)
plt.axvline(thr,color='k',ls='--',label='threshold'); plt.legend(); plt.xlabel('damage score')
plt.title('training slice damage scores'); plt.show()
print(f'infested slices above threshold: {(inf_scores>thr).mean():.1%}')

## Step 4 — curate training slices + QC

In [ ]:
rows=sc.curate_training(train, thr, min_keep=CFG.curation_min_keep)
man=pd.DataFrame(rows)
kept=man[man.kept].groupby('fruit_id').size(); tot=man.groupby('fruit_id').size()
summary=pd.DataFrame({'kept':kept,'total':tot}).fillna(0).astype(int)
summary['label']=[by[i].label for i in summary.index]
print('kept-slice summary (infested fruit):')
display(summary[summary.label==1].sort_values('kept').head(12))
flagged=man[man.reason.str.contains('early_detection')].fruit_id.unique()
print(f'\ninfested fruit flagged as early-detection (topk fallback): {len(flagged)}')
print(list(flagged))

## Step 5 — visual review (kept vs dropped) for one infested fruit

In [ ]:
from PIL import Image
ex=[f for f in tr_inf if f.fruit_id in set(man[man.label==1].fruit_id)]
if ex:
    f=ex[0]; sub=man[(man.fruit_id==f.fruit_id)].sort_values('score',ascending=False)
    show=pd.concat([sub[sub.kept].head(4), sub[~sub.kept].tail(4)])
    fig,ax=plt.subplots(1,len(show),figsize=(2*len(show),2.2))
    for a,(_,r) in zip(np.atleast_1d(ax),show.iterrows()):
        a.imshow(Image.open(r.slice),cmap='gray'); a.axis('off')
        a.set_title(f"{'KEEP' if r.kept else 'drop'}\n{r.score:.3f}",fontsize=8)
    fig.suptitle(f'{f.fruit_id}: highest-scoring (kept) vs lowest (dropped)'); plt.show()

## Step 6 — save manifest (editable) and build curated tree
Edit the `kept` column in `curation_manifest.csv` to override any decision, then re-run the last cell to rebuild the tree from the edited manifest.

In [ ]:
man.to_csv(CFG.out_dir/'curation_manifest.csv',index=False)
print('saved curation_manifest.csv — review/override the kept column if desired')

In [ ]:
# (re-)build curated tree from the (possibly edited) manifest
man=pd.read_csv(CFG.out_dir/'curation_manifest.csv')
curated_root=CFG.out_dir/CFG.curated_dirname
n=sc.build_curated_tree(man.to_dict('records'), CFG.data_root, curated_root)
print(f'curated tree written to {curated_root}  ({n} kept slices linked)')
print('>>> set CFG.curated_root = ', repr(str(curated_root)), 'in config.py before 02_train')